In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/cs-training.csv", index_col=0)

In [18]:
X = df.drop(columns="SeriousDlqin2yrs")
y = df["SeriousDlqin2yrs"]

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [20]:
print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())
print(X_train["MonthlyIncome"].isna().mean(),
      X_test["MonthlyIncome"].isna().mean())

(120000, 10) (30000, 10)
0.06684166666666666 0.06683333333333333
0.19729166666666667 0.20186666666666667


In [21]:
X_train = X_train.copy()
X_test = X_test.copy()

In [22]:
for part in (X_train, X_test):
    part["IncomeMissing"] = part["MonthlyIncome"].isna().astype(int)  

In [23]:
income_median = X_train["MonthlyIncome"].median()
dependents_median = X_train["NumberOfDependents"].median()

print(income_median, dependents_median)

5390.0 0.0


In [24]:
for part in (X_train, X_test):
    part["MonthlyIncome"] = part["MonthlyIncome"].fillna(income_median)
    part["NumberOfDependents"] = part["NumberOfDependents"].fillna(dependents_median)

print(X_train.isna().sum().sum(), X_test.isna().sum().sum())

0 0


In [25]:
print(X_train["IncomeMissing"].mean(), X_test["IncomeMissing"].mean())

0.19729166666666667 0.20186666666666667


In [26]:
past_due = ["NumberOfTime30-59DaysPastDueNotWorse",
            "NumberOfTime60-89DaysPastDueNotWorse",
            "NumberOfTimes90DaysLate"]

codes = X_train[past_due].isin([96, 98])
print(codes.sum(axis=1).value_counts())

has_code = codes.any(axis=1)
print(y_train.groupby(has_code).agg(["mean", "count"]))

0    119786
3       214
Name: count, dtype: int64
           mean   count
False  0.065984  119786
True   0.546729     214


## Past-due columns: the special codes 96 and 98

The three past-due columns (30-59 days, 60-89 days, 90+ days) count how many times a borrower was late in the last 2 years. In the data they contain the values 96 and 98, which are not real counts: no values appear between 17 and 96, and nobody is realistically 90+ days late 96 times in 2 years. The same rows carry the code in all three columns, so they most likely are placeholders for "unknown" (the dataset does not document them).

Why it matters: logistic regression would read 98 as "98 late payments", the worst possible borrower, and these values would distort the model.

What I do:
1. Count how many of the 3 columns hold a code in each row, using the training data only.
2. Compare the default rate of rows with a code against rows without (overall: 6.68%).
3. Decision: rows with a code default at 54.7% against 6.6% for the rest, so the code carries a strong signal. I add a flag column (PastDueCode) and replace the codes with the training median of the real values (0), so the model does not read 98 as "98 late payments".

The replacement value is computed on the training data only and applied to both train and test, to avoid data leakage.

In [27]:
for part in (X_train, X_test):
    part["PastDueCode"] = part[past_due].isin([96, 98]).any(axis=1).astype(int)
past_due_medians = {}
for c in past_due:
    valid = X_train[c][~X_train[c].isin([96, 98])]
    past_due_medians[c] = valid.median()
print(past_due_medians) 
for part in (X_train, X_test):
    for c in past_due:
        part[c] = part[c].replace([96, 98], past_due_medians[c])
print(X_train[past_due].max())
print(X_train["PastDueCode"].sum(), X_test["PastDueCode"].sum())

{'NumberOfTime30-59DaysPastDueNotWorse': np.float64(0.0), 'NumberOfTime60-89DaysPastDueNotWorse': np.float64(0.0), 'NumberOfTimes90DaysLate': np.float64(0.0)}
NumberOfTime30-59DaysPastDueNotWorse    13
NumberOfTime60-89DaysPastDueNotWorse    11
NumberOfTimes90DaysLate                 17
dtype: int64
214 55


In [28]:
print((X_train["age"] < 18).sum())
print(X_train["age"].describe())

1
count    120000.000000
mean         52.287842
std          14.771274
min           0.000000
25%          41.000000
50%          52.000000
75%          63.000000
max         109.000000
Name: age, dtype: float64


In [29]:
util = X_train["RevolvingUtilizationOfUnsecuredLines"]
print((util > 1).sum(), (util > 10).sum())
print(y_train.groupby(util > 1).agg(["mean", "count"]))

2640 190
                                         mean   count
RevolvingUtilizationOfUnsecuredLines                 
False                                 0.05991  117360
True                                  0.37500    2640


In [30]:
debt = X_train["DebtRatio"]
print(debt.quantile([0.5, 0.9, 0.95, 0.99]))
print(y_train.groupby(debt > 10).agg(["mean", "count"]))
print(X_train.groupby("IncomeMissing")["DebtRatio"].median())

0.50       0.366194
0.90    1263.000000
0.95    2441.000000
0.99    4977.000000
Name: DebtRatio, dtype: float64
               mean  count
DebtRatio                 
False      0.069663  96981
True       0.054955  23019
IncomeMissing
0       0.296108
1    1163.000000
Name: DebtRatio, dtype: float64


In [31]:
# age: impossible ages (< 18) -> training median of valid ages
age_median = X_train.loc[X_train["age"] >= 18, "age"].median()

# utilization: cap at the training 99th percentile
util_col = "RevolvingUtilizationOfUnsecuredLines"
util_cap = X_train[util_col].quantile(0.99)

# DebtRatio: median and cap learned from rows where income is known
known = X_train["IncomeMissing"] == 0
debt_median = X_train.loc[known, "DebtRatio"].median()
debt_cap = X_train.loc[known, "DebtRatio"].quantile(0.98)

# income: cap at the training 99th percentile
income_cap = X_train.loc[known, "MonthlyIncome"].quantile(0.99)

print(age_median, util_cap, debt_median, debt_cap, income_cap)

# apply the same rules to both parts

for part in (X_train, X_test):
    part.loc[part["age"] < 18, "age"] = age_median

    part["OverLimit"] = (part[util_col] > 1).astype(int)   # flag before capping
    part[util_col] = part[util_col].clip(upper=util_cap)

    part.loc[part["IncomeMissing"] == 1, "DebtRatio"] = debt_median
    part["DebtRatio"] = part["DebtRatio"].clip(upper=debt_cap)

    part["MonthlyIncome"] = part["MonthlyIncome"].clip(upper=income_cap)

print(X_train.describe().T)
print(X_test.describe().T)

52.0 1.0944386634299974 0.296108291 4.8380135539603 25000.0
                                         count         mean          std  \
RevolvingUtilizationOfUnsecuredLines  120000.0     0.319799     0.351879   
age                                   120000.0    52.288275    14.770503   
NumberOfTime30-59DaysPastDueNotWorse  120000.0     0.245342     0.696948   
DebtRatio                             120000.0     0.418566     0.652098   
MonthlyIncome                         120000.0  6151.521700  3919.512756   
NumberOfOpenCreditLinesAndLoans       120000.0     8.465692     5.161693   
NumberOfTimes90DaysLate               120000.0     0.090208     0.481895   
NumberRealEstateLoansOrLines          120000.0     1.018167     1.133055   
NumberOfTime60-89DaysPastDueNotWorse  120000.0     0.065125     0.331564   
NumberOfDependents                    120000.0     0.737592     1.107299   
IncomeMissing                         120000.0     0.197292     0.397956   
PastDueCode                 

In [32]:
big  = (X_train["IncomeMissing"] == 0) & (X_train["DebtRatio"] > 10)
print(big.sum())
print(X_train.loc[big, "MonthlyIncome"].describe())
print(y_train.groupby(big).agg(["mean", "count"]))

print(X_train.max().round(2))
print(X_test.max().round(2))

0
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: MonthlyIncome, dtype: float64
           mean   count
False  0.066842  120000
RevolvingUtilizationOfUnsecuredLines        1.09
age                                       109.00
NumberOfTime30-59DaysPastDueNotWorse       13.00
DebtRatio                                   4.84
MonthlyIncome                           25000.00
NumberOfOpenCreditLinesAndLoans            58.00
NumberOfTimes90DaysLate                    17.00
NumberRealEstateLoansOrLines               54.00
NumberOfTime60-89DaysPastDueNotWorse       11.00
NumberOfDependents                         20.00
IncomeMissing                               1.00
PastDueCode                                 1.00
OverLimit                                   1.00
dtype: float64
RevolvingUtilizationOfUnsecuredLines        1.09
age                                       109.00
NumberOfTime30-59DaysPastDueNotWorse        9.00
DebtRatio  

In [34]:
print(debt_cap)
print(X_train["DebtRatio"].max(), X_test["DebtRatio"].max())

4.8380135539603
4.8380135539603 4.8380135539603


## 02_baseline: data preparation

Goal: prepare the data for a baseline logistic regression without leaking information from the test set.

1. Split 80/20, stratified on the default flag.
2. Learn every cleaning value (medians, caps) from the training data only, then apply it to both parts.
3. Missing income and dependents: filled with the training median. Flag column IncomeMissing keeps the information.
4. Codes 96/98 in the past-due columns: placeholders, not real counts. Flag column PastDueCode, then replaced with the training median (0).
5. Extreme values:
Checked on the training data only:
- age: 1 borrower under 18 (age 0) -> replaced with the training median age.
- RevolvingUtilizationOfUnsecuredLines: 2,640 rows above 1 (over the credit limit) default at 37.5% against 6.0%, so this is a real signal. I add a flag OverLimit (utilization above 1) and cap the values at the training 99th percentile (1.09). Values like 50,708 are errors.
- DebtRatio: for rows with missing income the column holds a raw amount, not a ratio (median 1,163 against 0.30 where income is known). I replace those values with the training median of the real ratios (0.296); the flag IncomeMissing keeps the information.
- DebtRatio, known-income rows: 1,714 rows have a ratio above 10 because their income is about 0. A cap at the 99th percentile (682) was still absurd, so I cap at the training 98th percentile (4.84).
- MonthlyIncome: capped at the training 99th percentile of known incomes.

Quantile caps are learned from the training data only and applied to both train and test. No rows are deleted.

In [35]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [36]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_s, y_train)
dummy_pred = dummy.predict(X_test_s)

print("accuracy:", accuracy_score(y_test, dummy_pred))
print("recall:", recall_score(y_test, dummy_pred))

accuracy: 0.9331666666666667
recall: 0.0


In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

proba = model.predict_proba(X_test_s)[:, 1]
pred = (proba >= 0.5).astype(int)

print("AUC:", roc_auc_score(y_test, proba))
print("precision:", precision_score(y_test, pred))
print("recall:", recall_score(y_test, pred))
print(confusion_matrix(y_test, pred))

AUC: 0.85741618448966
precision: 0.6113074204946997
recall: 0.17256857855361596
[[27775   220]
 [ 1659   346]]


In [38]:
import pandas as pd

weights = pd.Series(model.coef_[0], index=X_train.columns).sort_values()
print(weights)

age                                    -0.236569
MonthlyIncome                          -0.159255
DebtRatio                              -0.038977
IncomeMissing                          -0.017379
OverLimit                               0.038970
NumberOfDependents                      0.052414
PastDueCode                             0.096415
NumberRealEstateLoansOrLines            0.120183
NumberOfOpenCreditLinesAndLoans         0.147085
NumberOfTime60-89DaysPastDueNotWorse    0.203395
NumberOfTime30-59DaysPastDueNotWorse    0.289787
NumberOfTimes90DaysLate                 0.318624
RevolvingUtilizationOfUnsecuredLines    0.699260
dtype: float64


In [39]:
for t in [0.5, 0.3, 0.2, 0.15, 0.1]:
    pred_t = (proba >= t).astype(int)
    print(t,
          "flagged:", pred_t.sum(),
          "precision:", round(precision_score(y_test, pred_t), 3),
          "recall:", round(recall_score(y_test, pred_t), 3))

0.5 flagged: 566 precision: 0.611 recall: 0.173
0.3 flagged: 1171 precision: 0.52 recall: 0.304
0.2 flagged: 1859 precision: 0.448 recall: 0.415
0.15 flagged: 2729 precision: 0.38 recall: 0.517
0.1 flagged: 4893 precision: 0.266 recall: 0.648


Threshold trade-off (test set): lowering the threshold from 0.5 to 0.2 raises recall from 0.17 to 0.42 while precision falls from 0.61 to 0.45; at 0.1 recall is 0.65 but only 27% of flagged borrowers really default. The AUC (0.857) does not depend on the threshold. The final threshold should be chosen on a validation split or a cost rule, not on the test set.

## Baseline model

1. Scaling: each column is transformed to (value - mean) / standard deviation, with mean and standard deviation learned from the training data only, so that no column dominates because of its units.
2. Reference model: "predict nobody defaults" reaches accuracy 0.933 with recall 0, which shows that accuracy is misleading with 6.7% defaults.
3. Logistic regression, fitted on the training data, evaluated once on the test data:
   AUC 0.857, precision 0.611, recall 0.173 (threshold 0.5).
   Confusion matrix: 27,775 true negatives, 220 false positives, 1,659 false negatives, 346 true positives.
4. Weights: the strongest risk drivers are credit utilization, the number of late payments (90+ days the highest), and, in the other direction, age and income. The flag PastDueCode looks small only because it is almost always 0; per borrower with the code it multiplies the odds of default by roughly 10.

The AUC is good, but recall is low because the 0.5 threshold is high for a 6.7% default rate. Choosing the threshold is the next improvement.